In [1]:
!unzip /content/text_classification_dataset.zip

Archive:  /content/text_classification_dataset.zip
   creating: text_classification_dataset/
  inflating: text_classification_dataset/train_split.csv  
  inflating: __MACOSX/text_classification_dataset/._train_split.csv  
  inflating: text_classification_dataset/val_split.csv  
  inflating: __MACOSX/text_classification_dataset/._val_split.csv  


In [2]:
!unzip /content/hindi_corpus.zip

Streaming output truncated to the last 5000 lines.
  inflating: hindi_corpus/val/139496.txt  
  inflating: __MACOSX/hindi_corpus/val/._139496.txt  
  inflating: hindi_corpus/val/138950.txt  
  inflating: __MACOSX/hindi_corpus/val/._138950.txt  
  inflating: hindi_corpus/val/131974.txt  
  inflating: __MACOSX/hindi_corpus/val/._131974.txt  
  inflating: hindi_corpus/val/131784.txt  
  inflating: __MACOSX/hindi_corpus/val/._131784.txt  
  inflating: hindi_corpus/val/134824.txt  
  inflating: __MACOSX/hindi_corpus/val/._134824.txt  
  inflating: hindi_corpus/val/133193.txt  
  inflating: __MACOSX/hindi_corpus/val/._133193.txt  
  inflating: hindi_corpus/val/135284.txt  
  inflating: __MACOSX/hindi_corpus/val/._135284.txt  
  inflating: hindi_corpus/val/138978.txt  
  inflating: __MACOSX/hindi_corpus/val/._138978.txt  
  inflating: hindi_corpus/val/136955.txt  
  inflating: __MACOSX/hindi_corpus/val/._136955.txt  
  inflating: hindi_corpus/val/136799.txt  
  inflating: __MACOSX/hindi_corpu

In [2]:
import os
import random
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = torch.cuda.is_available()
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


class Config:
    # Model settings for the small Hindi GPT backbone
    vocab_size = 5000
    block_size = 128
    n_embd = 384
    n_head = 6
    n_layer = 6
    dropout = 0.1
    pad_id = 0
    unk_id = 1
    bos_id = 2
    eos_id = 3


# File paths
HINDI_PATH = "hindi_corpus/train"
CLASS_PATH = "text_classification_dataset/train.csv"


def train_hindi_tokenizer(folder_path):
    input_files_list = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            input_files_list.append(os.path.join(folder_path, filename))

    input_files = ",".join(input_files_list)

    # Train SentencePiece with explicit special token ids so padding is clean
    spm.SentencePieceTrainer.train(
        input=input_files,
        model_prefix="hindi_bpe",
        vocab_size=Config.vocab_size,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=Config.pad_id,
        unk_id=Config.unk_id,
        bos_id=Config.bos_id,
        eos_id=Config.eos_id,
        hard_vocab_limit=False,
    )


def load_hindi_tokenizer():
    sp = spm.SentencePieceProcessor()
    sp.load("hindi_bpe.model")
    return sp


class HindiLMDataset(Dataset):
    def __init__(self, folder_path, sp, block_size=128):
        super().__init__()
        self.files = [
            os.path.join(folder_path, filename)
            for filename in os.listdir(folder_path)
            if filename.endswith(".txt")
        ]
        self.sp = sp
        self.block_size = block_size

    def __len__(self):
        return len(self.files) * 3

    def __getitem__(self, idx):
        # Sample a random contiguous chunk from a random text file
        while True:
            file_path = self.files[idx % len(self.files)]

            with open(file_path, encoding="utf-8") as f:
                text = f.read().strip()

            tokens = self.sp.encode(text)

            if len(tokens) < self.block_size + 1:
                idx = random.randint(0, len(self.files) - 1)
                continue

            start = random.randint(0, len(tokens) - self.block_size - 1)
            end = start + self.block_size

            x = tokens[start:start + self.block_size]
            y = tokens[start + 1:end + 1]
            return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def load_classification_dataset(csv_path):
    df = pd.read_csv(csv_path)

    texts = df.iloc[:, 0].astype(str).tolist()
    labels_raw = df.iloc[:, 1].tolist()

    # Factorize labels so the code works for integers, strings, or mixed label formats
    labels, uniques = pd.factorize(labels_raw)
    label_to_id = {label: int(i) for i, label in enumerate(uniques.tolist())}

    return texts, labels.tolist(), label_to_id


def tokenize_classification_dataset(sp, texts, max_len=128):
    out = []
    for text in texts:
        ids = sp.encode(text)
        ids = ids[:max_len]
        ids += [Config.pad_id] * (max_len - len(ids))
        out.append(ids)
    return out


In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.scale = math.sqrt(self.head_dim)

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)

    def forward(self, x):
        batch_size, seq_len, channels = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / self.scale
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~causal_mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)

        return self.proj(context)


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(4 * config.n_embd, config.n_embd),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.attn = MultiHeadAttention(config)
        self.ff = FeedForward(config)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x


class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*[Block(config) for _ in range(config.n_layer)])
        self.ln = nn.LayerNorm(config.n_embd)

    def forward(self, idx):
        batch_size, seq_len = idx.shape
        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(seq_len, device=idx.device))

        x = tok + pos
        x = self.dropout(x)
        x = self.blocks(x)
        return self.ln(x)


class GPTLanguageModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = MiniGPT(config)
        self.head = nn.Linear(config.n_embd, config.vocab_size)

    def forward(self, x, y=None):
        h = self.backbone(x)
        logits = self.head(h)

        loss = None
        if y is not None:
            batch_size, seq_len, vocab_size = logits.shape
            loss = F.cross_entropy(logits.view(batch_size * seq_len, vocab_size), y.view(batch_size * seq_len))

        return logits, loss


def train_model(model, train_dataset, val_dataset=None, epochs=15, batch_size=32, lr=3e-4):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    if val_dataset is not None:
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits, loss = model(x, y)

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        if val_dataset is not None:
            model.eval()
            total_val_loss = 0

            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device)

                with torch.no_grad():
                    logits, loss = model(x, y)

                total_val_loss += loss.item()

            avg_val_loss = total_val_loss / len(val_loader)

            print(f"Epoch {epoch+1}")
            print(f"Train Loss: {avg_train_loss:.4f}")
            print(f"Val Loss: {avg_val_loss:.4f}\n")

        else:
            print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f}")


def compute_perplexity(model, dataset, batch_size=32):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        num_workers=2 if torch.cuda.is_available() else 0,
    )
    model.eval()

    total_loss = 0.0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            _, loss = model(x, y)
            total_loss += loss.item()

    return math.exp(total_loss / len(loader))


def generate(model, sp, start_text, max_tokens=50):
    model.eval()
    ids = sp.encode(start_text)
    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_tokens):
        logits, _ = model(x)
        last_logits = logits[:, -1, :]
        probs = F.softmax(last_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_token], dim=1)

        if x.size(1) > Config.block_size:
            x = x[:, -Config.block_size:]

    return sp.decode(x[0].tolist())


In [11]:
class GPTClassifier(nn.Module):
    def __init__(self, gpt_model, config, num_classes=3):
        super().__init__()
        self.backbone = gpt_model.backbone
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(config.n_embd, num_classes)

    def forward(self, x, y=None):
        h = self.backbone(x)

        # Mean pool only over real tokens, not padding
        mask = (x != Config.pad_id).unsqueeze(-1).float()
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        logits = self.fc(self.dropout(pooled))
        loss = None
        if y is not None:
            loss = F.cross_entropy(logits, y)
        return logits, loss


class ClassificationDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.long),
        )


def train_classifier(model, train_dataset, val_dataset=None, epochs=8, batch_size=64):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    if val_dataset is not None:
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

    for param in model.backbone.parameters():
        param.requires_grad = False

    for param in model.backbone.blocks[-1].parameters():
        param.requires_grad = True
    for param in model.backbone.ln.parameters():
        param.requires_grad = True

    optimizer = torch.optim.AdamW(
        [
            {"params": model.fc.parameters(), "lr": 3e-4},
            {"params": model.backbone.blocks[-1].parameters(), "lr": 1e-5},
            {"params": model.backbone.ln.parameters(), "lr": 1e-5},
        ]
    )

    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            _, loss = model(x, y)

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        if val_dataset is not None:
            model.eval()
            total_val_loss = 0

            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(device)
                    y = y.to(device)

                    _, loss = model(x, y)
                    total_val_loss += loss.item()

            avg_val_loss = total_val_loss / len(val_loader)

            print(f"Epoch {epoch+1}")
            print(f"Train Loss: {avg_train_loss:.4f}")
            print(f"Val Loss: {avg_val_loss:.4f}\n")

        else:
            print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f}")


def evaluate(model, dataset, batch_size=64):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        num_workers=2 if torch.cuda.is_available() else 0,
    )

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits, _ = model(x)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total


In [ ]:
HINDI_PATH_train= "hindi_corpus/train"
HINDI_PATH_val = "hindi_corpus/val"
CLASS_PATH = "text_classification_dataset/train.csv"
CLASS_PATH = "text_classification_dataset/train.csv"
if not os.path.exists("hindi_bpe.model"):
    train_hindi_tokenizer(HINDI_PATH_train)

sp = load_hindi_tokenizer()
train_dataset = HindiLMDataset(HINDI_PATH_train, sp, Config.block_size)
val_dataset = HindiLMDataset(HINDI_PATH_val, sp, Config.block_size)

lm_model = GPTLanguageModel(Config()).to(device)

print("Training LM:")
train_model(lm_model, train_dataset, val_dataset, epochs=15)

train_ppl = compute_perplexity(lm_model, train_dataset)
val_ppl = compute_perplexity(lm_model, val_dataset)

print("\nTrain Perplexity:", train_ppl)
print("Validation Perplexity:", val_ppl)

generated_text = generate(lm_model, sp, "भारत")
print("\nGenerated Text:", generated_text)

with open("generated_text.txt", "w", encoding="utf-8") as f:
    f.write(generated_text)

torch.save(lm_model.state_dict(), "gpt_lm.pth")


train_texts, train_labels, label_map = load_classification_dataset("text_classification_dataset/train.csv")
val_texts, val_labels, _ = load_classification_dataset("text_classification_dataset/val.csv")

X_train_tok = tokenize_classification_dataset(sp, train_texts, Config.block_size)
X_val_tok = tokenize_classification_dataset(sp, val_texts, Config.block_size)

train_ds = ClassificationDataset(X_train_tok, train_labels)
val_ds = ClassificationDataset(X_val_tok, val_labels)

classifier = GPTClassifier(lm_model, Config(), num_classes=len(label_map)).to(device)

print("\nTraining Classifier:")
train_classifier(classifier, train_ds, val_ds, epochs=300, batch_size=64)

acc = evaluate(classifier, val_ds)
print("\nValidation Accuracy:", acc)

torch.save(classifier.state_dict(), "gpt_classifier.pth")

Training LM:


In [ ]:
lm_model = GPTLanguageModel(Config()).to(device)
lm_model.load_state_dict(torch.load("gpt_lm.pth", map_location=device))
lm_model.eval()

sp = load_hindi_tokenizer()

train_dataset = HindiLMDataset(HINDI_PATH_train, sp, Config.block_size)
val_dataset = HindiLMDataset(HINDI_PATH_val, sp, Config.block_size)

train_ppl = compute_perplexity(lm_model, train_dataset)
val_ppl = compute_perplexity(lm_model, val_dataset)

print("Train Perplexity:", train_ppl)
print("Validation Perplexity:", val_ppl)

generated_text = generate(lm_model, sp, "भारत")
print("\nGenerated Text:\n", generated_text)

with open("generated_text.txt", "w", encoding="utf-8") as f:
    f.write(generated_text)

train_texts, train_labels, label_map = load_classification_dataset("text_classification_dataset/train.csv")
val_texts, val_labels, _ = load_classification_dataset("text_classification_dataset/val.csv")

classifier = GPTClassifier(lm_model, Config(), num_classes=len(label_map)).to(device)
classifier.load_state_dict(torch.load("gpt_classifier.pth", map_location=device))
classifier.eval()

X_val_tok = tokenize_classification_dataset(sp, val_texts, Config.block_size)
val_ds = ClassificationDataset(X_val_tok, val_labels)

acc = evaluate(classifier, val_ds)
print("\nValidation Accuracy:", acc)

correct_samples = []

for i in range(len(val_ds)):
    x, y = val_ds[i]

    x_tensor = x.unsqueeze(0).to(device)

    with torch.no_grad():
        logits, _ = classifier(x_tensor)
        pred = torch.argmax(logits, dim=1)

    if pred.item() == y:
        decoded_text = sp.decode(x.tolist())
        correct_samples.append((decoded_text, y, pred.item()))

    if len(correct_samples) == 5:
        break

with open("correct_predictions.txt", "w", encoding="utf-8") as f:
    for i, (text, label, pred) in enumerate(correct_samples):
        f.write(f"Sample {i+1}\n")
        f.write(f"True Label: {label}, Predicted: {pred}\n")
        f.write(f"Review: {text}\n")
        f.write("\n" + "-" * 50 + "\n\n")

Perplexity: 10.991272966403006

Generated Text:
 भारत का मूल निवासी था। अमोनिया का राजपरिवार ने 15 मई 1868 से 15 दिसम्बर 1866 तक शासन किया। इसके फलस्वरूप इंग्लैंड के राजा को राणा वर्मा में रखा गया एवं वे राणाएं का निर्माण


/tmp/ipykernel_4325/1035273124.py:113: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  labels, uniques = pd.factorize(labels_raw)



Accuracy: 0.3611111111111111
